# Fase II: Preparación y Exploración de Datos

**Proyecto Integrador - Inteligencia Computacional**  
**Dominio:** Medio Ambiente &rarr; Calidad y Seguridad del Agua (*Water Quality*)  
**Integrantes:** Melanie Tomalá, Jean Cedeño, Damian Torres  
**Docente:** Ing. Daniel Quirumbay  

---

## Objetivos de esta fase (según rúbrica de cátedra)
1. **Documentar:** procedencia, tamaño, variables, valores faltantes, duplicados, distribución de clases y posibles sesgos.
2. **Realizar visualizaciones pertinentes:** y justificar limpieza, codificación, escalado y selección de variables.
3. **Separar entrenamiento y prueba ANTES de ajustar transformaciones:** usar `Pipeline` de scikit-learn para evitar fuga de información (*data leakage*).

**Experimento cerrado:** consultar `reports/` y el notebook de fase IV para revisar resultados. No reejecutar preparación o selección sobre este test ya evaluado.


## 1. Importación de Librerías y Configuración del Entorno

In [ ]:
import os
from pathlib import Path
if not Path('data').exists() and Path('../data').exists():
    os.chdir('..')
from pathlib import Path
if Path("reports/phase4_selection.json").exists():
    raise RuntimeError("Experimento cerrado. Consultar los reportes guardados; no sobrescribir resultados.")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Fijar semilla determinista universal exigida por la rúbrica
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="muted")
print("Librerías importadas correctamente con RANDOM_STATE = 42")

## 2. Carga del Dataset y Auditoría de Integridad

In [ ]:
data_path = "data/water_quality.csv"
df_raw = pd.read_csv(data_path)

print(f"Dimensiones iniciales del dataset: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")
df_raw.head()

## 3. Detección y Saneamiento de Datos Corruptos (`#NUM!`)

El dataset presenta anomalías reales en la columna `ammonia` y en la variable objetivo `is_safe`, donde se introdujeron cadenas `#NUM!` procedentes de errores de cálculo en hoja de cálculo original. Se procede a su identificación y descarte justificado.

In [ ]:
# Identificar filas con valores corruptos
corrupt_mask = (df_raw['ammonia'] == '#NUM!') | (df_raw['is_safe'] == '#NUM!')
print(f"Cantidad de registros corruptos identificados: {corrupt_mask.sum()}")
print("Índices de filas corruptas:", df_raw[corrupt_mask].index.tolist())

# Descartar registros corruptos y convertir tipos
df_clean = df_raw[~corrupt_mask].copy()
df_clean['ammonia'] = pd.to_numeric(df_clean['ammonia'], errors='raise')
df_clean['is_safe'] = df_clean['is_safe'].astype(int)

print(f"\nRegistros limpios finales: {len(df_clean)}")
print(f"Valores nulos en el dataset saneado: {df_clean.isnull().sum().sum()}")

## 4. Análisis de Desbalance de Clases

Analizamos la proporción de muestras de agua no segura (`0`) frente a agua segura (`1`).

In [ ]:
counts = df_clean['is_safe'].value_counts()
ratios = df_clean['is_safe'].value_counts(normalize=True)

print(f"Clase 0 (No segura): {counts[0]} muestras ({ratios[0]*100:.2f}%)")
print(f"Clase 1 (Segura):    {counts[1]} muestras ({ratios[1]*100:.2f}%)")

fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(['No segura (0)', 'Segura (1)'], [counts[0], counts[1]], color=['#D9534F', '#5CB85C'], width=0.45)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 80, f"{h:,}\n({h/len(df_clean)*100:.1f}%)", ha='center', va='bottom', fontweight='bold')
ax.set_title('Distribución de Clases - Seguridad del Agua', fontsize=12, fontweight='bold')
ax.set_ylabel('Frecuencia absoluta')
ax.set_ylim(0, counts[0]*1.15)
plt.tight_layout()
plt.show()

**Conclusión del desbalance:** Al tener un 88.6% de clase negativa, un modelo trivial que prediga siempre `0` obtendría 88.6% de exactitud pero no reconocería ninguna muestra de la clase segura (Recall de clase 1 = 0%). Esto justifica formalmente el requisito de la cátedra de evaluar mediante **Macro-F1, Precision, Recall y ROC-AUC**.

## 5. Correlaciones y Concentración de Contaminantes

In [ ]:
corr = df_clean.corr(method='spearman')
target_corr = corr['is_safe'].drop('is_safe').sort_values()

plt.figure(figsize=(9, 6))
colors = ['#D9534F' if v < 0 else '#5CB85C' for v in target_corr.values]
plt.barh(target_corr.index, target_corr.values, color=colors, height=0.6)
plt.axvline(0, color='gray', linestyle='--')
plt.title('Correlación de Spearman de Contaminantes con la Seguridad del Agua', fontweight='bold')
plt.xlabel('Coeficiente de Correlación')
plt.tight_layout()
plt.show()

## 6. Partición Estratificada sin Fuga de Información (*No Data Leakage*)

Separamos en 80% entrenamiento y 20% prueba con estratificación por clase. El ajuste de cualquier transformador se efectúa **únicamente sobre `X_train`**.

In [ ]:
features = [c for c in df_clean.columns if c != 'is_safe']
X = df_clean[features]
y = df_clean['is_safe']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"X_train shape: {X_train.shape} | Proporción seguros: {y_train.mean()*100:.2f}%")
print(f"X_test shape:  {X_test.shape} | Proporción seguros: {y_test.mean()*100:.2f}%")

# Construir pipeline con StandardScaler
preprocessor = Pipeline([
    ('scaler', StandardScaler())
])

# Calibrar (fit) EXCLUSIVAMENTE sobre X_train
preprocessor.fit(X_train)
print("\nPreprocesador ajustado exitosamente sin consultar datos de prueba.")

## 7. Persistencia de Artefactos de la Fase II

In [ ]:
os.makedirs('data/processed', exist_ok=True)

train_df = X_train.copy()
train_df['is_safe'] = y_train
test_df = X_test.copy()
test_df['is_safe'] = y_test

train_df.to_csv('data/processed/train.csv', index=False)
test_df.to_csv('data/processed/test.csv', index=False)
dump(preprocessor, 'data/processed/preprocessor.joblib')

print("Artefactos exportados:")
print(" - data/processed/train.csv (6,396 filas)")
print(" - data/processed/test.csv  (1,600 filas)")
print(" - data/processed/preprocessor.joblib")